# Analysis: Economic Indicators & Presidential Approval

This notebook explores the relationship between economic variables (CPI, Unemployment, Gas Prices) and Presidential Approval ratings.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import altair as alt
from statsmodels.tsa.stattools import adfuller
import numpy as np

# Check if data exists, if not, try to fetch it or warn user
try:
    df = pd.read_csv('../data/processed_data.csv', index_col=0, parse_dates=True)
    df.index.name = 'date'
    print("Data loaded successfully.")
    print(df.head())
except FileNotFoundError:
    print("Data file not found. Please run 'src/fetch_data.py' first.")

## 2. Visual Inspection
Let's look at the raw time series to identify trends and seasonality.

In [ ]:
# Melt dataframe for easier Altair plotting
df_long = df.reset_index().melt('date', var_name='indicator', value_name='value')

base = alt.Chart(df_long).mark_line().encode(
    x='date:T',
    y=alt.Y('value:Q', scale=alt.Scale(zero=False)),
    color='indicator:N',
    tooltip=['date', 'indicator', 'value']
 ).properties(
    width=600,
    height=150
 )

chart = base.facet(
    row='indicator:N',
    resolve=alt.Resolve(scale=alt.AxisResolveMap(y=alt.ResolveMode('independent')))
 ).interactive()

chart

## 3. Stationarity Tests (ADF)
Granger Causality requires stationary time series (constant mean/variance over time). We use the Augmented Dickey-Fuller test.

In [ ]:
def adf_test(series, title=''):
    print(f'Augmented Dickey-Fuller Test: {title}')
    if series.nunique() <= 1:
        print(f"Series {title} is constant. Skipping ADF test.\n")
        return
    result = adfuller(series.dropna(), autolag='AIC')
    labels = ['ADF Test Statistic','p-value','# Lags Used','Number of Observations Used']
    out = pd.Series(result[0:4], index=labels)
    for key,val in result[4].items():
        out[f'Critical Value ({key})'] = val
    print(out)
    print('')
    if result[1] <= 0.05:
        print("Strong evidence against the None hypothesis(Ho), reject the None hypothesis. Data has no unit root and is stationary")
    else:
        print("Weak evidence against None hypothesis, time series has a unit root, indicating it is non-stationary ")

for col in df.columns:
    adf_test(df[col], title=col)

## 4. Differencing
Most economic data is non-stationary. We'll take the first difference (t - t-1) to stabilize the mean.

In [ ]:
df_diff = df.diff()

print("Running ADF on Differenced Data:")
for col in df_diff.columns:
    adf_test(df_diff[col], title=col)

## 5. Cross-Correlation Analysis
Checking correlations at different lags.

In [ ]:
# Example: Cross correlation between Gas Prices and Approval
target = 'approval_rating'
predictor = 'gas_prices'

# Debug: Check data availability
print(f"Correlation Analysis: {predictor} vs {target}")
print(f"N records {predictor}: {df_diff[predictor].count()}")
print(f"N records {target}: {df_diff[target].count()}")

lags = np.arange(-12, 13)
corrs = [df_diff[target].corr(df_diff[predictor].shift(lag)) for lag in lags]

corr_df = pd.DataFrame({'lag': lags, 'correlation': corrs})
print("Correlation results (head):")
print(corr_df.head())

alt.Chart(corr_df.dropna()).mark_bar().encode(
    x='lag:O',
    y='correlation:Q',
    color=alt.condition(
        alt.datum.correlation > 0,
        alt.value('steelblue'),  # The positive color
        alt.value('orange')      # The negative color
    )
 ).properties(title=f'Cross Correlation: {predictor} vs {target}')